In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from utils import ASSETS_DIR
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
import numpy as np





In [ ]:
df = pd.read_parquet(ASSETS_DIR / 'nacc74_cleaned.parquet')
df.describe()

In [ ]:


# 1. Separiamo la matrice delle Feature (X) dal Target (y)
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# 2. TRAIN-TEST SPLIT (80% Train, 20% Test)
# stratify=y è fondamentale per mantenere le stesse percentuali di malati nei due set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Buchi (NaN) iniziali in X_train: {X_train.isna().sum().sum()}")

# 3. CONFIGURAZIONE DEL KNN IMPUTER
# weights='distance' dà più importanza ai vicini più vicini (più simili)
imputer = KNNImputer(n_neighbors=5, weights='distance')

# 4. ADDESTRAMENTO E TRASFORMAZIONE
# Il modello "impara" le distribuzioni SOLO da X_train
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Il modello applica quanto imparato su X_test (senza barare)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 5. ARROTONDAMENTO PER DATI CLINICI DISCRETI
# Riportiamo le medie del KNN a numeri interi (es. 0.66 diventa 1.0)
X_train_imp = X_train_imp.round()
X_test_imp = X_test_imp.round()

print(f"Buchi (NaN) finali in X_train: {X_train_imp.isna().sum().sum()}")
print(f"Buchi (NaN) finali in X_test: {X_test_imp.isna().sum().sum()}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# 1. MIN-MAX SCALING (Deve avvenire SEMPRE prima della PCA)
scaler = MinMaxScaler()

# Il modello impara i minimi e massimi SOLO dal Train Set
X_train_scaled = scaler.fit_transform(X_train_imp)
# Scala il Test Set basandosi su quanto imparato (No Leakage)
X_test_scaled = scaler.transform(X_test_imp)

# 2. APPLICAZIONE DELLA PCA
# Invece di scegliere un numero fisso di componenti a caso, chiediamo a scikit-learn
# di tenere un numero di componenti sufficiente a spiegare il 90% della varianza totale.
pca = PCA(n_components=0.90, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Dimensioni originali: {X_train_scaled.shape[1]} features")
print(f"Dimensioni dopo PCA: {X_train_pca.shape[1]} componenti principali (spiegano il 90% della varianza)")

# 3. VISUALIZZAZIONE DELLA VARIANZA SPIEGATA
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.title("Varianza Cumulativa Spiegata dalle Componenti Principali")
plt.xlabel("Numero di Componenti")
plt.ylabel("Varianza Spiegata")
plt.grid(True)
plt.show()

In [ ]:

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def train_model(X_train, y_train, model):
    

    #
    scoring_metrics = {
        'precision': make_scorer(precision_score, average='macro', zero_division=0),
        'recall': make_scorer(recall_score, average='macro', zero_division=0),
        'f1_score': make_scorer(f1_score, average='macro', zero_division=0),
        'auc_roc': 'roc_auc_ovr'  # One-vs-Rest per gestire l'AUC a 3 classi
    }

    # 4. ESECUZIONE DELLA K-FOLD (Sul Train Set!)
    # NB: Usa i dati scalati ma con i nomi delle colonne cliniche intatte
    print("Addestramento in corso (5-Fold CV)...")
    cv_results = cross_validate(
        estimator=model,
        X=X_train, 
        y=y_train,
        cv=cv_strategy,
        scoring=scoring_metrics,
        return_train_score=False
    )

    # 5. REPORT DEI RISULTATI
    print("\n=== RISULTATI RANDOM FOREST (Media su 5 Folds) ===")
    print(f"Precision (Macro): {np.mean(cv_results['test_precision']):.4f} ± {np.std(cv_results['test_precision']):.4f}")
    print(f"Recall (Macro):    {np.mean(cv_results['test_recall']):.4f} ± {np.std(cv_results['test_recall']):.4f}")
    print(f"F1-Score (Macro):  {np.mean(cv_results['test_f1_score']):.4f} ± {np.std(cv_results['test_f1_score']):.4f}")
    print(f"AUC-ROC (OvR):     {np.mean(cv_results['test_auc_roc']):.4f} ± {np.std(cv_results['test_auc_roc']):.4f}")

In [ ]:
rf_model = RandomForestClassifier(
        n_estimators=100, 
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )

logistic_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='newton-cholesky')

mlp = MLPClassifier(
    hidden_layer_sizes=(50, 30),
    solver='adam',
    max_iter=2000,
    random_state=42,
    learning_rate_init=0.001,
    early_stopping=True,
    n_iter_no_change=20,
    learning_rate='adaptive'
)

train_model(X_train_scaled, y_train, rf_model)
train_model(X_train_pca, y_train, logistic_model)
train_model(X_train_pca, y_train, mlp)
    

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

def generate_oof_predictions(X_train_scaled, y_train, model):

    print("Generazione previsioni Out-of-Fold (Validazione Interna in corso)...")

    # 1. Generiamo le previsioni "cieche" su tutto il Train Set tramite la 5-Fold
    y_pred_oof = cross_val_predict(
    estimator=rf_model,
    X=X_train_scaled, 
    y=y_train,
    cv=cv_strategy,  # Usiamo esattamente la stessa k-fold stratificata di prima
    n_jobs=-1
    )

    # 2. Calcoliamo la Matrice di Confusione INTERNA
    cm_oof = confusion_matrix(y_train, y_pred_oof)

    # 3. Visualizzazione grafica
    plt.figure(figsize=(8, 6))
    etichette = ['Sani (0)', 'Alzheimer (1)', 'Lewy Body (2)']

    sns.heatmap(cm_oof, annot=True, fmt='d', cmap='Oranges', # Colore diverso per distinguerla dal Test
            xticklabels=etichette, 
            yticklabels=etichette,
            linewidths=1, linecolor='black')

    plt.title('Matrice di Confusione (Validazione Interna Out-of-Fold)', fontsize=14, pad=15)
    plt.ylabel('Diagnosi Reale (Medico)', fontsize=12, fontweight='bold')
    plt.xlabel(f'Previsione ({type(model).__name__})', fontsize=12, fontweight='bold')
    plt.show()

    # 4. Report clinico della validazione
    print("\n" + "="*50)
    print("CLASSIFICATION REPORT (VALIDAZIONE INTERNA - 5 FOLD)")
    print("="*50)
    print(classification_report(y_train, y_pred_oof, target_names=etichette))

In [ ]:
generate_oof_predictions(X_train_scaled, y_train, rf_model)
generate_oof_predictions(X_train_pca, y_train, logistic_model)
generate_oof_predictions(X_train_pca, y_train, mlp)